# stage 0

In [3]:
# ============================================================
# Stage 0. Setup paths and load raw PySPoC meta-analysis input
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

INPUT_CSV = Path("/Users/ilg/Desktop/year4/M4R/python_files/metadata/meta_raw_unnormalised/synthetic_raw_statistics.csv")

PROJECT_DIR = Path("/Users/ilg/Desktop/year4/M4R/analysis")
OUTPUT_DIR = PROJECT_DIR / "eda_output_unnormalised"

STAGE0_DIR = OUTPUT_DIR / "00_input"
STAGE0_DIR.mkdir(parents=True, exist_ok=True)

RAW_COPY_PATH = STAGE0_DIR / "syn_raw_input_copy.csv"
STAGE0_OUTPUT_PATH = STAGE0_DIR / "syn_stage0_with_metadata.csv"

# Important: PySPoC writes two header rows:
# row 1 = Statistic / ReducedStatistic name
# row 2 = Reducer name, or self/self_1/self_2/... for ReducedStatistics
df_raw = pd.read_csv(INPUT_CSV, header=[0, 1])

print(f"Loaded file: {INPUT_CSV}")
print(f"Raw shape after two-row header read: {df_raw.shape}")

df_raw.head()

Loaded file: /Users/ilg/Desktop/year4/M4R/python_files/metadata/meta_raw_unnormalised/synthetic_raw_statistics.csv
Raw shape after two-row header read: (1044, 1232)


dataset_id                                          file_stem  \
  Unnamed: 0_level_1                                 Unnamed: 1_level_1   
0                  1  synth_0001_additive_K2_C0p75_O0p00_very_sparse...   
1                  2  synth_0002_additive_K2_C0p75_O0p00_very_sparse...   
2                  3  synth_0003_additive_K2_C0p75_O0p00_very_sparse...   
3                  4  synth_0004_additive_K2_C0p75_O0p00_sparse_loca...   
4                  5  synth_0005_additive_K2_C0p75_O0p00_sparse_loca...   

        dataset_type       model_family         n_clusters     contrast_level  \
  Unnamed: 2_level_1 Unnamed: 3_level_1 Unnamed: 4_level_1 Unnamed: 5_level_1   
0         structured           additive                  2               0.75   
1         structured           additive                  2               0.75   
2         structured           additive                  2               0.75   
3         structured           additive                  2               0.75   
4         structured           additive                  2               0.75   

   row_overlap_ratio  col_overlap_ratio         shape_name       row_fraction  \
  Unnamed: 6_level_1 Unnamed: 7_level_1 Unnamed: 8_level_1 Unnamed: 9_level_1   
0                0.0                0.0  very_sparse_local               0.03   
1                0.0                0.0  very_sparse_local               0.03   
2                0.0                0.0  very_sparse_local               0.03   
3                0.0                0.0       sparse_local               0.05   
4                0.0                0.0       sparse_local               0.05   

   ... pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3  \
   ...                                                          self_294   
0  ...                                          -0.031686                  
1  ...                                          -0.084886                  
2  ...                                          -0.132975                  
3  ...                                           0.033049                  
4  ...                                           0.139781                  

                                                               \
   self_295  self_296  self_297  self_298  self_299  self_300   
0 -0.066972  0.042134  0.072826  0.273599 -0.083447 -0.028012   
1 -0.026331 -0.085074 -0.189125  0.047166  0.076350  0.120494   
2  0.110938  0.099337 -0.057622  0.130140  0.020781 -0.040326   
3 -0.037108 -0.016836  0.037337 -0.154369 -0.196687  0.116028   
4 -0.003720 -0.085929  0.034785  0.054987  0.148512  0.096955   

  pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3  \
                                                                         self_1   
0                                           0.016583                              
1                                           0.017190                              
2                                           0.017107                              
3                                           0.016781                              
4                                           0.017070                              

                       
     self_2    self_3  
0  0.016287  0.015982  
1  0.016684  0.016411  
2  0.016547  0.016150  
3  0.016352  0.015969  
4  0.016686  0.016329  

[5 rows x 1232 columns]

In [4]:
# ============================================================
# Stage 0.1 Save a raw copy
# ============================================================

# Save exactly what we loaded, still with the MultiIndex columns.
df_raw.to_csv(RAW_COPY_PATH, index=False)
print(f"Raw input copy saved to: {RAW_COPY_PATH}")

Raw input copy saved to: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/00_input/syn_raw_input_copy.csv


In [5]:
# ============================================================
# Stage 0.2 Flatten PySPoC two-level columns
# ============================================================

def clean_header_part(x):
    x = str(x)

    # Pandas creates names like "Unnamed: 0_level_1" for blank cells
    if x.startswith("Unnamed:"):
        return ""

    return x.strip()


def flatten_pyspoc_column(col_tuple):
    top, bottom = col_tuple

    top = clean_header_part(top)
    bottom = clean_header_part(bottom)

    # Metadata columns usually have only the top-level name.
    if bottom == "":
        return top

    # If top and bottom are identical, avoid duplication.
    if top == bottom:
        return top

    # Normal PySPoC metric column:
    # Statistic + Reducer, or ReducedStatistic + self/self_1/self_2
    return f"{top}.{bottom}"


df = df_raw.copy()
df.columns = [flatten_pyspoc_column(col) for col in df.columns]

print(f"Shape after flattening columns: {df.shape}")
print(df.columns[:20].tolist())
df.head()

Shape after flattening columns: (1044, 1232)
['dataset_id', 'file_stem', 'dataset_type', 'model_family', 'n_clusters', 'contrast_level', 'row_overlap_ratio', 'col_overlap_ratio', 'shape_name', 'row_fraction', 'col_fraction', 'seed', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_1', 'pyspoc.statistics.basic.Covariance.feature_c

,dataset_id,file_stem,dataset_type,model_family,n_clusters,contrast_level,row_overlap_ratio,col_overlap_ratio,shape_name,row_fraction,...,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_294,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_295,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_296,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_297,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_298,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_299,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_300,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_1,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_2,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_3
0,1,synth_0001_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,-0.031686,-0.066972,0.042134,0.072826,0.273599,-0.083447,-0.028012,0.016583,0.016287,0.015982
1,2,synth_0002_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,-0.084886,-0.026331,-0.085074,-0.189125,0.047166,0.076350,0.120494,0.017190,0.016684,0.016411
2,3,synth_0003_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,-0.132975,0.110938,0.099337,-0.057622,0.130140,0.020781,-0.040326,0.017107,0.016547,0.016150
3,4,synth_0004_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,sparse_local,0.05,...,0.033049,-0.037108,-0.016836,0.037337,-0.154369,-0.196687,0.116028,0.016781,0.016352,0.015969
4,5,synth_0005_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,sparse_local,0.05,...,0.139781,-0.003720,-0.085929,0.034785,0.054987,0.148512,0.096955,0.017070,0.016686,0.016329


In [6]:
# ============================================================
# Stage 0.3 Remove accidental empty columns
# ============================================================

empty_named_cols = [col for col in df.columns if col == ""]
if empty_named_cols:
    df = df.drop(columns=empty_named_cols)

print(f"Shape after dropping empty-named columns: {df.shape}")

Shape after dropping empty-named columns: (1044, 1232)


In [7]:
# ============================================================
# Stage 0.4 Add fallback dataset identifier if needed
# ============================================================

# Depending on how the raw file was written, metadata columns should already exist.
# If not, create simple identifiers.

if "file_stem" not in df.columns:
    df.insert(0, "file_stem", [f"dataset_{i+1:04d}" for i in range(len(df))])

if "dataset_id" not in df.columns:
    df.insert(0, "dataset_id", np.arange(1, len(df) + 1))

print(df[["dataset_id", "file_stem"]].head())

   dataset_id                                          file_stem
0           1  synth_0001_additive_K2_C0p75_O0p00_very_sparse...
1           2  synth_0002_additive_K2_C0p75_O0p00_very_sparse...
2           3  synth_0003_additive_K2_C0p75_O0p00_very_sparse...
3           4  synth_0004_additive_K2_C0p75_O0p00_sparse_loca...
4           5  synth_0005_additive_K2_C0p75_O0p00_sparse_loca...


In [8]:
# ============================================================
# Stage 0.5 Add derived synthetic-design metadata
# ============================================================

# Make sure these columns are numeric if present.
for col in ["n_clusters", "contrast_level", "row_overlap_ratio",
            "col_overlap_ratio", "row_fraction", "col_fraction", "seed"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if {"row_fraction", "col_fraction"}.issubset(df.columns):
    df["cell_fraction"] = df["row_fraction"] * df["col_fraction"]
else:
    df["cell_fraction"] = np.nan

if {"n_clusters", "row_fraction", "col_fraction"}.issubset(df.columns):
    df["approx_total_planted_fraction"] = (
        df["n_clusters"] * df["row_fraction"] * df["col_fraction"]
    )
else:
    df["approx_total_planted_fraction"] = np.nan

if {"row_overlap_ratio", "col_overlap_ratio"}.issubset(df.columns):
    df["mean_overlap_ratio"] = (
        df["row_overlap_ratio"] + df["col_overlap_ratio"]
    ) / 2
else:
    df["mean_overlap_ratio"] = np.nan

preview_cols = [
    "file_stem",
    "model_family",
    "n_clusters",
    "contrast_level",
    "shape_name",
    "row_fraction",
    "col_fraction",
    "cell_fraction",
    "approx_total_planted_fraction",
    "seed",
]

preview_cols = [col for col in preview_cols if col in df.columns]
df[preview_cols].head()

,file_stem,model_family,n_clusters,contrast_level,shape_name,row_fraction,col_fraction,cell_fraction,approx_total_planted_fraction,seed
0,synth_0001_additive_K2_C0p75_O0p00_very_sparse...,additive,2,0.75,very_sparse_local,0.03,0.08,0.0024,0.0048,0
1,synth_0002_additive_K2_C0p75_O0p00_very_sparse...,additive,2,0.75,very_sparse_local,0.03,0.08,0.0024,0.0048,1
2,synth_0003_additive_K2_C0p75_O0p00_very_sparse...,additive,2,0.75,very_sparse_local,0.03,0.08,0.0024,0.0048,2
3,synth_0004_additive_K2_C0p75_O0p00_sparse_loca...,additive,2,0.75,sparse_local,0.05,0.10,0.0050,0.0100,0
4,synth_0005_additive_K2_C0p75_O0p00_sparse_loca...,additive,2,0.75,sparse_local,0.05,0.10,0.0050,0.0100,1


In [9]:
# ============================================================
# Stage 0.6 Identify metadata columns and metric columns
# ============================================================

metadata_cols = [
    "dataset_id",
    "file_stem",
    "dataset_type",
    "model_family",
    "n_clusters",
    "contrast_level",
    "row_overlap_ratio",
    "col_overlap_ratio",
    "mean_overlap_ratio",
    "shape_name",
    "row_fraction",
    "col_fraction",
    "cell_fraction",
    "approx_total_planted_fraction",
    "background_noise_std",
    "placement",
    "size_jitter",
    "noise_distribution",
    "seed",
    "has_planted_biclusters",
]

metadata_cols = [col for col in metadata_cols if col in df.columns]
metric_cols = [col for col in df.columns if col not in metadata_cols]

print(f"Number of metadata columns: {len(metadata_cols)}")
print(f"Number of metric/statistic columns: {len(metric_cols)}")

print("\nFirst few metric columns:")
print(metric_cols[:10])

Number of metadata columns: 15
Number of metric/statistic columns: 1220

First few metric columns:
['pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matri

In [10]:
# ============================================================
# Stage 0.7 Basic row-level sanity checks
# ============================================================

print("Number of datasets:", len(df))

if "file_stem" in df.columns:
    print("Unique file_stem values:", df["file_stem"].nunique())

if "model_family" in df.columns:
    print("\nModel family counts:")
    print(df["model_family"].value_counts(dropna=False))

if "n_clusters" in df.columns:
    print("\nTrue K counts:")
    print(df["n_clusters"].value_counts(dropna=False).sort_index())

if "shape_name" in df.columns:
    print("\nShape counts:")
    print(df["shape_name"].value_counts(dropna=False))

if "seed" in df.columns:
    print("\nSeed counts:")
    print(df["seed"].value_counts(dropna=False).sort_index())

Number of datasets: 1044
Unique file_stem values: 1044

Model family counts:
model_family
additive          513
multiplicative    513
NaN                18
Name: count, dtype: int64

True K counts:
n_clusters
0     18
2    432
3    378
5    216
Name: count, dtype: int64

Shape counts:
shape_name
very_sparse_local     162
sparse_local          162
medium_balanced       162
large_balanced        162
large_row_dominant    108
large_col_dominant    108
huge_balanced         108
very_huge_balanced     54
NaN                    18
Name: count, dtype: int64

Seed counts:
seed
0    345
1    345
2    345
3      3
4      3
5      3
Name: count, dtype: int64


In [11]:
# ============================================================
# Stage 0.8 Save Stage 0 output with normal one-row columns
# ============================================================

df.to_csv(STAGE0_OUTPUT_PATH, index=False)

print(f"Stage 0 output saved to: {STAGE0_OUTPUT_PATH}")
print(f"Final Stage 0 shape: {df.shape}")

Stage 0 output saved to: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/00_input/syn_stage0_with_metadata.csv
Final Stage 0 shape: (1044, 1235)


In [12]:
df = pd.read_csv(STAGE0_OUTPUT_PATH)

# stage 1

In [13]:
# ============================================================
# Stage 1. Load Stage 0 output
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/Users/ilg/Desktop/year4/M4R/python_files")
STAGE1_DIR = OUTPUT_DIR / "01_split_metadata_features"
STAGE1_DIR.mkdir(parents=True, exist_ok=True)

METADATA_OUTPUT_PATH = STAGE1_DIR / "metadata.csv"
FEATURES_RAW_OUTPUT_PATH = STAGE1_DIR / "features_raw.csv"
COLUMN_REPORT_PATH = STAGE1_DIR / "stage1_column_report.csv"

df = pd.read_csv(STAGE0_OUTPUT_PATH)

print(f"Loaded Stage 0 file: {STAGE0_OUTPUT_PATH}")
print(f"Shape: {df.shape}")
df.head()

Loaded Stage 0 file: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/00_input/syn_stage0_with_metadata.csv
Shape: (1044, 1235)


,dataset_id,file_stem,dataset_type,model_family,n_clusters,contrast_level,row_overlap_ratio,col_overlap_ratio,shape_name,row_fraction,...,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_297,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_298,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_299,pyspoc.rstatistics.pca.PCAEigenVectors.pca_eigenvectors_top_1_2_3.self_300,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_1,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_2,pyspoc.rstatistics.pca.PCAVarianceExplainedRatio.pca_variance_ratio_top_1_2_3.self_3,cell_fraction,approx_total_planted_fraction,mean_overlap_ratio
0,1,synth_0001_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,0.072826,0.273599,-0.083447,-0.028012,0.016583,0.016287,0.015982,0.0024,0.0048,0.0
1,2,synth_0002_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,-0.189125,0.047166,0.076350,0.120494,0.017190,0.016684,0.016411,0.0024,0.0048,0.0
2,3,synth_0003_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,very_sparse_local,0.03,...,-0.057622,0.130140,0.020781,-0.040326,0.017107,0.016547,0.016150,0.0024,0.0048,0.0
3,4,synth_0004_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,sparse_local,0.05,...,0.037337,-0.154369,-0.196687,0.116028,0.016781,0.016352,0.015969,0.0050,0.0100,0.0
4,5,synth_0005_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,sparse_local,0.05,...,0.034785,0.054987,0.148512,0.096955,0.017070,0.016686,0.016329,0.0050,0.0100,0.0


In [14]:
# ============================================================
# Stage 1.1 Define metadata columns
# ============================================================

metadata_cols = [
    "dataset_id",
    "file_stem",
    "dataset_type",
    "model_family",
    "n_clusters",
    "contrast_level",
    "row_overlap_ratio",
    "col_overlap_ratio",
    "mean_overlap_ratio",
    "shape_name",
    "row_fraction",
    "col_fraction",
    "cell_fraction",
    "approx_total_planted_fraction",
    "background_noise_std",
    "placement",
    "size_jitter",
    "noise_distribution",
    "seed",
    "has_planted_biclusters",
]

metadata_cols = [col for col in metadata_cols if col in df.columns]

feature_cols = [col for col in df.columns if col not in metadata_cols]

print(f"Metadata columns: {len(metadata_cols)}")
print(f"Feature/statistic columns: {len(feature_cols)}")

print("\nMetadata columns:")
print(metadata_cols)

print("\nFirst 10 feature columns:")
print(feature_cols[:10])

Metadata columns: 15
Feature/statistic columns: 1220

Metadata columns:
['dataset_id', 'file_stem', 'dataset_type', 'model_family', 'n_clusters', 'contrast_level', 'row_overlap_ratio', 'col_overlap_ratio', 'mean_overlap_ratio', 'shape_name', 'row_fraction', 'col_fraction', 'cell_fraction', 'approx_total_planted_fraction', 'seed']

First 10 feature columns:
['pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1', 'pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2', 'pyspoc.statistics.basic.Covariance.

In [15]:
# ============================================================
# Stage 1.2 Split into metadata and raw features
# ============================================================

metadata_df = df[metadata_cols].copy()
features_raw = df[feature_cols].copy()

print("metadata_df shape:", metadata_df.shape)
print("features_raw shape:", features_raw.shape)

metadata_df shape: (1044, 15)
features_raw shape: (1044, 1220)


In [16]:
# ============================================================
# Stage 1.3 Convert feature columns to numeric
# ============================================================
# PySPoC outputs should be numeric, but CSV reading can sometimes
# keep columns as object if there are strings, inf, or formatting issues.

features_numeric = features_raw.apply(pd.to_numeric, errors="coerce")

non_numeric_report = []

for col in features_raw.columns:
    original_non_missing = features_raw[col].notna().sum()
    numeric_non_missing = features_numeric[col].notna().sum()
    newly_coerced_to_nan = original_non_missing - numeric_non_missing

    non_numeric_report.append({
        "column": col,
        "original_dtype": str(features_raw[col].dtype),
        "numeric_dtype": str(features_numeric[col].dtype),
        "original_non_missing": original_non_missing,
        "numeric_non_missing": numeric_non_missing,
        "newly_coerced_to_nan": newly_coerced_to_nan,
    })

non_numeric_report = pd.DataFrame(non_numeric_report)

print("Feature matrix after numeric conversion:", features_numeric.shape)
print("Columns with values coerced to NaN:")
display(non_numeric_report[non_numeric_report["newly_coerced_to_nan"] > 0].head(20))

Feature matrix after numeric conversion: (1044, 1220)
Columns with values coerced to NaN:


,column,original_dtype,numeric_dtype,original_non_missing,numeric_non_missing,newly_coerced_to_nan


In [17]:
# ============================================================
# Stage 1.4 Basic sanity checks
# ============================================================

print("Number of datasets:", len(metadata_df))

if "file_stem" in metadata_df.columns:
    print("Unique file_stem:", metadata_df["file_stem"].nunique())

if "dataset_id" in metadata_df.columns:
    print("Unique dataset_id:", metadata_df["dataset_id"].nunique())

print("\nFeature matrix shape:")
print(features_numeric.shape)

print("\nTotal missing feature values:")
print(features_numeric.isna().sum().sum())

print("\nTotal infinite feature values:")
print(np.isinf(features_numeric.to_numpy(dtype=float)).sum())

Number of datasets: 1044
Unique file_stem: 1044
Unique dataset_id: 1044

Feature matrix shape:
(1044, 1220)

Total missing feature values:
0

Total infinite feature values:
0


In [18]:
# ============================================================
# Stage 1.5 Create a column report
# ============================================================

column_report = []

for col in metadata_cols:
    column_report.append({
        "column": col,
        "column_type": "metadata",
        "dtype": str(df[col].dtype),
        "non_missing": df[col].notna().sum(),
        "missing": df[col].isna().sum(),
    })

for col in features_numeric.columns:
    values = features_numeric[col]
    column_report.append({
        "column": col,
        "column_type": "feature",
        "dtype": str(values.dtype),
        "non_missing": values.notna().sum(),
        "missing": values.isna().sum(),
    })

column_report = pd.DataFrame(column_report)

column_report.head()

,column,column_type,dtype,non_missing,missing
0,dataset_id,metadata,int64,1044,0
1,file_stem,metadata,object,1044,0
2,dataset_type,metadata,object,1026,18
3,model_family,metadata,object,1026,18
4,n_clusters,metadata,int64,1044,0


In [19]:
# ============================================================
# Stage 1.6 Save Stage 1 outputs
# ============================================================

metadata_df.to_csv(METADATA_OUTPUT_PATH, index=False)
features_numeric.to_csv(FEATURES_RAW_OUTPUT_PATH, index=False)
column_report.to_csv(COLUMN_REPORT_PATH, index=False)

print(f"Metadata saved to: {METADATA_OUTPUT_PATH}")
print(f"Raw numeric features saved to: {FEATURES_RAW_OUTPUT_PATH}")
print(f"Column report saved to: {COLUMN_REPORT_PATH}")

Metadata saved to: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/01_split_metadata_features/metadata.csv
Raw numeric features saved to: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/01_split_metadata_features/features_raw.csv
Column report saved to: /Users/ilg/Desktop/year4/M4R/analysis/eda_output_unnormalised/01_split_metadata_features/stage1_column_report.csv


In [20]:
# ============================================================
# Stage 1.7 Preview
# ============================================================

display(metadata_df.head())
display(features_numeric.iloc[:5, :10])

,dataset_id,file_stem,dataset_type,model_family,n_clusters,contrast_level,row_overlap_ratio,col_overlap_ratio,mean_overlap_ratio,shape_name,row_fraction,col_fraction,cell_fraction,approx_total_planted_fraction,seed
0,1,synth_0001_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,0
1,2,synth_0002_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,1
2,3,synth_0003_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,2
3,4,synth_0004_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,0.0,sparse_local,0.05,0.10,0.0050,0.0100,0
4,5,synth_0005_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,0.0,sparse_local,0.05,0.10,0.0050,0.0100,1


,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_3,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_4,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_5
0,23218.24947,0.920464,0.963210,1.673357,1.667996,0.009265,0.010102,0.010561,0.010970,0.010471
1,140935.60930,0.896379,1.038905,1.718792,1.673097,0.008807,0.011738,0.011252,0.010261,0.012559
2,26050.64587,1.098125,0.979163,1.727451,1.678585,0.012850,0.010654,0.009472,0.010486,0.010647
3,13511.40929,1.040181,1.059388,1.735619,1.672523,0.011651,0.012193,0.010575,0.011296,0.009690
4,102570.77300,1.024567,0.994082,1.712905,1.685830,0.011309,0.010998,0.011195,0.012048,0.013005


In [21]:
print("metadata shape:", metadata_df.shape)
print("features shape:", features_numeric.shape)

print("unique file_stem:", metadata_df["file_stem"].nunique())
print("unique dataset_id:", metadata_df["dataset_id"].nunique())

display(metadata_df.head())
display(features_numeric.iloc[:5, :5])

metadata shape: (1044, 15)
features shape: (1044, 1220)
unique file_stem: 1044
unique dataset_id: 1044


,dataset_id,file_stem,dataset_type,model_family,n_clusters,contrast_level,row_overlap_ratio,col_overlap_ratio,mean_overlap_ratio,shape_name,row_fraction,col_fraction,cell_fraction,approx_total_planted_fraction,seed
0,1,synth_0001_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,0
1,2,synth_0002_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,1
2,3,synth_0003_additive_K2_C0p75_O0p00_very_sparse...,structured,additive,2,0.75,0.0,0.0,0.0,very_sparse_local,0.03,0.08,0.0024,0.0048,2
3,4,synth_0004_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,0.0,sparse_local,0.05,0.10,0.0050,0.0100,0
4,5,synth_0005_additive_K2_C0p75_O0p00_sparse_loca...,structured,additive,2,0.75,0.0,0.0,0.0,sparse_local,0.05,0.10,0.0050,0.0100,1


,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2
0,23218.24947,0.920464,0.963210,1.673357,1.667996
1,140935.60930,0.896379,1.038905,1.718792,1.673097
2,26050.64587,1.098125,0.979163,1.727451,1.678585
3,13511.40929,1.040181,1.059388,1.735619,1.672523
4,102570.77300,1.024567,0.994082,1.712905,1.685830


# stage 2

In [22]:
# ============================================================
# Stage 2. Load Stage 1 outputs
# ============================================================

PROJECT_DIR = Path("/Users/ilg/Desktop/year4/M4R/python_files")
OUTPUT_DIR = PROJECT_DIR / "eda_output_unnormalised"

STAGE1_DIR = OUTPUT_DIR / "01_split_metadata_features"
METADATA_PATH = STAGE1_DIR / "metadata.csv"
FEATURES_RAW_PATH = STAGE1_DIR / "features_raw.csv"

STAGE2_DIR = OUTPUT_DIR / "02_feature_quality_control"
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_QC_REPORT_PATH = STAGE2_DIR / "feature_qc_report.csv"
FEATURES_FILTERED_RAW_PATH = STAGE2_DIR / "features_filtered_raw.csv"
METADATA_STAGE2_PATH = STAGE2_DIR / "metadata_stage2.csv"
DROPPED_FEATURES_PATH = STAGE2_DIR / "dropped_features.csv"
KEPT_FEATURES_PATH = STAGE2_DIR / "kept_features.csv"

metadata_df = pd.read_csv(METADATA_PATH)
features_raw = pd.read_csv(FEATURES_RAW_PATH)

print("metadata shape:", metadata_df.shape)
print("features_raw shape:", features_raw.shape)

metadata shape: (1044, 15)
features_raw shape: (1044, 1220)


In [23]:
# ============================================================
# Stage 2.1 Make sure feature matrix is numeric
# ============================================================

features_numeric = features_raw.apply(pd.to_numeric, errors="coerce")

# Replace +/- inf with NaN for quality control.
features_clean_for_qc = features_numeric.replace([np.inf, -np.inf], np.nan)

print("features_numeric shape:", features_numeric.shape)
print("Total NaN after numeric conversion:", features_clean_for_qc.isna().sum().sum())
print("Total infinite values before replacement:", np.isinf(features_numeric.to_numpy(dtype=float)).sum())

features_numeric shape: (1044, 1220)
Total NaN after numeric conversion: 0
Total infinite values before replacement: 0


In [24]:
# ============================================================
# Stage 2.2 Build feature-level QC report
# ============================================================

n_datasets = len(features_clean_for_qc)
qc_rows = []

for col in features_clean_for_qc.columns:
    x = features_clean_for_qc[col]
    x_nonmissing = x.dropna()

    missing_count = x.isna().sum()
    missing_fraction = missing_count / n_datasets

    if len(x_nonmissing) == 0:
        q1 = q3 = iqr = np.nan
        xmin = xmax = mean = median = std = np.nan
        n_unique = 0
        value_range = np.nan
        range_to_iqr = np.nan
    else:
        q1 = x_nonmissing.quantile(0.25)
        q3 = x_nonmissing.quantile(0.75)
        iqr = q3 - q1
        xmin = x_nonmissing.min()
        xmax = x_nonmissing.max()
        mean = x_nonmissing.mean()
        median = x_nonmissing.median()
        std = x_nonmissing.std()
        n_unique = x_nonmissing.nunique()
        value_range = xmax - xmin
        range_to_iqr = value_range / (iqr + 1e-12)

    qc_rows.append({
        "feature": col,
        "n_datasets": n_datasets,
        "missing_count": missing_count,
        "missing_fraction": missing_fraction,
        "non_missing_count": len(x_nonmissing),
        "n_unique": n_unique,
        "min": xmin,
        "q1": q1,
        "median": median,
        "q3": q3,
        "max": xmax,
        "mean": mean,
        "std": std,
        "iqr": iqr,
        "range": value_range,
        "range_to_iqr": range_to_iqr,
    })

feature_qc = pd.DataFrame(qc_rows)

feature_qc.head()

,feature,n_datasets,missing_count,missing_fraction,non_missing_count,n_unique,min,q1,median,q3,max,mean,std,iqr,range,range_to_iqr
0,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,2.386392e-11,0.041549,14.559809,814.967844,862960.422402,6554.202230,38468.346034,814.926295,862960.422402,1058.942911
1,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,420,8.940858e-01,0.982250,1.030267,1.174808,10.195631,1.309297,0.831713,0.192558,9.301545,48.305272
2,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,391,9.204675e-01,0.979163,1.025078,1.157933,9.742403,1.303091,0.803770,0.178771,8.821935,49.347799
3,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.665369e+00,2.931296,6.948123,23.066052,225.840371,18.029591,29.263261,20.134756,224.175002,11.133733
4,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.618522e+00,2.354643,5.024400,14.776044,164.413872,12.595349,19.770196,12.421401,162.795351,13.106037


In [25]:
# ============================================================
# Stage 2.3 Define objective filtering rules
# ============================================================

MISSING_FRACTION_THRESHOLD = 0.05
NEAR_ZERO_STD_THRESHOLD = 1e-12
NEAR_ZERO_IQR_THRESHOLD = 1e-12
EXPLOSIVE_RANGE_TO_IQR_THRESHOLD = 1e6

feature_qc["flag_all_missing"] = feature_qc["non_missing_count"] == 0

feature_qc["flag_high_missing"] = (
    feature_qc["missing_fraction"] > MISSING_FRACTION_THRESHOLD
)

feature_qc["flag_constant"] = (
    feature_qc["n_unique"] <= 1
)

feature_qc["flag_near_constant_std"] = (
    feature_qc["std"].fillna(0) <= NEAR_ZERO_STD_THRESHOLD
)

feature_qc["flag_near_constant_iqr"] = (
    feature_qc["iqr"].fillna(0) <= NEAR_ZERO_IQR_THRESHOLD
)

feature_qc["flag_explosive_range"] = (
    feature_qc["range_to_iqr"].replace([np.inf, -np.inf], np.nan).fillna(0)
    > EXPLOSIVE_RANGE_TO_IQR_THRESHOLD
)

# For PCA, we should definitely remove missing/constant/near-constant features.
# Explosive features are flagged, but not automatically removed here.
# You can inspect them before deciding.
feature_qc["drop_for_stage3"] = (
    feature_qc["flag_all_missing"]
    | feature_qc["flag_high_missing"]
    | feature_qc["flag_constant"]
    | feature_qc["flag_near_constant_std"]
    | feature_qc["flag_near_constant_iqr"]
)

def make_drop_reason(row):
    reasons = []
    if row["flag_all_missing"]:
        reasons.append("all_missing")
    if row["flag_high_missing"]:
        reasons.append("high_missing")
    if row["flag_constant"]:
        reasons.append("constant")
    if row["flag_near_constant_std"]:
        reasons.append("near_zero_std")
    if row["flag_near_constant_iqr"]:
        reasons.append("near_zero_iqr")
    if row["flag_explosive_range"]:
        reasons.append("explosive_range_flag")
    return "; ".join(reasons)

feature_qc["qc_reason"] = feature_qc.apply(make_drop_reason, axis=1)

feature_qc.head()

,feature,n_datasets,missing_count,missing_fraction,non_missing_count,n_unique,min,q1,median,q3,...,range,range_to_iqr,flag_all_missing,flag_high_missing,flag_constant,flag_near_constant_std,flag_near_constant_iqr,flag_explosive_range,drop_for_stage3,qc_reason
0,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,2.386392e-11,0.041549,14.559809,814.967844,...,862960.422402,1058.942911,False,False,False,False,False,False,False,
1,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,420,8.940858e-01,0.982250,1.030267,1.174808,...,9.301545,48.305272,False,False,False,False,False,False,False,
2,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,391,9.204675e-01,0.979163,1.025078,1.157933,...,8.821935,49.347799,False,False,False,False,False,False,False,
3,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.665369e+00,2.931296,6.948123,23.066052,...,224.175002,11.133733,False,False,False,False,False,False,False,
4,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.618522e+00,2.354643,5.024400,14.776044,...,162.795351,13.106037,False,False,False,False,False,False,False,


In [26]:
# ============================================================
# Stage 2.4 Summarise QC results
# ============================================================

print("Total features:", len(feature_qc))
print("Features marked to drop before PCA:", int(feature_qc["drop_for_stage3"].sum()))
print("Features kept for Stage 3:", int((~feature_qc["drop_for_stage3"]).sum()))

print("\nFlag counts:")
flag_cols = [
    "flag_all_missing",
    "flag_high_missing",
    "flag_constant",
    "flag_near_constant_std",
    "flag_near_constant_iqr",
    "flag_explosive_range",
]

for col in flag_cols:
    print(f"{col}: {int(feature_qc[col].sum())}")

print("\nTop missing features:")
display(
    feature_qc.sort_values("missing_fraction", ascending=False)
    [["feature", "missing_count", "missing_fraction", "qc_reason"]]
    .head(20)
)

print("\nMost explosive features by range/IQR:")
display(
    feature_qc.sort_values("range_to_iqr", ascending=False)
    [["feature", "min", "median", "max", "iqr", "range_to_iqr", "qc_reason"]]
    .head(20)
)

print("\nNear-constant examples:")
display(
    feature_qc[feature_qc["flag_constant"] | feature_qc["flag_near_constant_iqr"]]
    [["feature", "n_unique", "std", "iqr", "min", "max", "qc_reason"]]
    .head(20)
)

Total features: 1220
Features marked to drop before PCA: 6
Features kept for Stage 3: 1214

Flag counts:
flag_all_missing: 0
flag_high_missing: 0
flag_constant: 4
flag_near_constant_std: 6
flag_near_constant_iqr: 6
flag_explosive_range: 3

Top missing features:


,feature,missing_count,missing_fraction,qc_reason
0,pyspoc.statistics.basic.Covariance.feature_cov...,0,0.0,
811,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
818,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
817,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
816,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
815,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
814,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
813,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
812,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,
810,pyspoc.statistics.basic.SpearmanR.feature_spea...,0,0.0,



Most explosive features by range/IQR:


,feature,min,median,max,iqr,range_to_iqr,qc_reason
648,pyspoc.statistics.basic.SpearmanR.feature_spea...,21490.620738,1.268038e+07,2.008927e+86,2.964603e+14,6.776378e+71,explosive_range_flag
432,pyspoc.statistics.basic.KendallTau.feature_ken...,72.528369,1.745381e+03,1.283576e+47,1.969808e+07,6.516250e+39,explosive_range_flag
216,pyspoc.statistics.basic.Precision.feature_prec...,0.000001,6.868472e-02,4.190426e+10,2.414250e+01,1.735705e+09,explosive_range_flag
129,pyspoc.statistics.basic.Covariance.feature_cov...,0.006110,1.074143e-02,2.091813e+02,6.759108e-03,3.094715e+04,
106,pyspoc.statistics.basic.Covariance.feature_cov...,0.005942,1.054257e-02,2.192573e+02,7.339228e-03,2.987390e+04,
186,pyspoc.statistics.basic.Covariance.feature_cov...,0.005715,1.051356e-02,2.140090e+02,7.258116e-03,2.948468e+04,
130,pyspoc.statistics.basic.Covariance.feature_cov...,0.006265,1.086501e-02,2.503313e+02,8.850155e-03,2.828482e+04,
160,pyspoc.statistics.basic.Covariance.feature_cov...,0.005340,1.100952e-02,3.345488e+02,1.202002e-02,2.783218e+04,
162,pyspoc.statistics.basic.Covariance.feature_cov...,0.005514,1.084883e-02,2.717454e+02,1.031359e-02,2.634775e+04,
107,pyspoc.statistics.basic.Covariance.feature_cov...,0.005972,1.051364e-02,1.841145e+02,7.088097e-03,2.597432e+04,



Near-constant examples:


,feature,n_unique,std,iqr,min,max,qc_reason
433,pyspoc.statistics.basic.KendallTau.feature_ken...,1,0.000000e+00,0.0,1.0,1.0,constant; near_zero_std; near_zero_iqr
434,pyspoc.statistics.basic.KendallTau.feature_ken...,1,0.000000e+00,0.0,1.0,1.0,constant; near_zero_std; near_zero_iqr
639,pyspoc.statistics.basic.KendallTau.feature_ken...,1,0.000000e+00,0.0,100.0,100.0,constant; near_zero_std; near_zero_iqr
649,pyspoc.statistics.basic.SpearmanR.feature_spea...,2,4.067542e-17,0.0,1.0,1.0,near_zero_std; near_zero_iqr
650,pyspoc.statistics.basic.SpearmanR.feature_spea...,2,3.572563e-17,0.0,1.0,1.0,near_zero_std; near_zero_iqr
855,pyspoc.statistics.basic.SpearmanR.feature_spea...,1,0.000000e+00,0.0,100.0,100.0,constant; near_zero_std; near_zero_iqr


In [27]:
# ============================================================
# Stage 2.5 Create filtered raw feature matrix
# ============================================================

kept_features = feature_qc.loc[~feature_qc["drop_for_stage3"], "feature"].tolist()
dropped_features = feature_qc.loc[feature_qc["drop_for_stage3"], "feature"].tolist()

features_filtered_raw = features_clean_for_qc[kept_features].copy()

print("Original feature matrix:", features_clean_for_qc.shape)
print("Filtered raw feature matrix:", features_filtered_raw.shape)

print("Remaining missing values in kept features:")
print(features_filtered_raw.isna().sum().sum())

Original feature matrix: (1044, 1220)
Filtered raw feature matrix: (1044, 1214)
Remaining missing values in kept features:
0


In [28]:
# ============================================================
# Stage 2.6 Save Stage 2 outputs
# ============================================================

feature_qc.to_csv(FEATURE_QC_REPORT_PATH, index=False)
features_filtered_raw.to_csv(FEATURES_FILTERED_RAW_PATH, index=False)
metadata_df.to_csv(METADATA_STAGE2_PATH, index=False)

feature_qc.loc[feature_qc["drop_for_stage3"]].to_csv(DROPPED_FEATURES_PATH, index=False)
feature_qc.loc[~feature_qc["drop_for_stage3"]].to_csv(KEPT_FEATURES_PATH, index=False)

print(f"Feature QC report saved to: {FEATURE_QC_REPORT_PATH}")
print(f"Filtered raw features saved to: {FEATURES_FILTERED_RAW_PATH}")
print(f"Metadata saved to: {METADATA_STAGE2_PATH}")
print(f"Dropped features saved to: {DROPPED_FEATURES_PATH}")
print(f"Kept features saved to: {KEPT_FEATURES_PATH}")

Feature QC report saved to: /Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/02_feature_quality_control/feature_qc_report.csv
Filtered raw features saved to: /Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/02_feature_quality_control/features_filtered_raw.csv
Metadata saved to: /Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/02_feature_quality_control/metadata_stage2.csv
Dropped features saved to: /Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/02_feature_quality_control/dropped_features.csv
Kept features saved to: /Users/ilg/Desktop/year4/M4R/python_files/eda_output_unnormalised/02_feature_quality_control/kept_features.csv


In [29]:
# ============================================================
# Stage 2.7 Quick preview of filtered matrix
# ============================================================

display(features_filtered_raw.iloc[:5, :10])
display(feature_qc.head())

,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Determinant.scaled_matrix_determinant,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Diag.matrix_diagonal_entries_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.EigenValues.matrix_eigenvalues_all_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_1,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_2,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_3,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_4,pyspoc.statistics.basic.Covariance.feature_covariance_matrix.pyspoc.reducers.basic.Moment.columnwise_moments_2_4_5
0,23218.249473,0.920464,0.963210,1.673357,1.667996,0.009265,0.010102,0.010561,0.010970,0.010471
1,140935.609329,0.896379,1.038905,1.718792,1.673097,0.008807,0.011738,0.011252,0.010261,0.012559
2,26050.645869,1.098125,0.979163,1.727451,1.678585,0.012850,0.010654,0.009472,0.010486,0.010647
3,13511.409292,1.040181,1.059388,1.735619,1.672523,0.011651,0.012193,0.010575,0.011296,0.009690
4,102570.772973,1.024567,0.994082,1.712905,1.685830,0.011309,0.010998,0.011195,0.012048,0.013005


,feature,n_datasets,missing_count,missing_fraction,non_missing_count,n_unique,min,q1,median,q3,...,range,range_to_iqr,flag_all_missing,flag_high_missing,flag_constant,flag_near_constant_std,flag_near_constant_iqr,flag_explosive_range,drop_for_stage3,qc_reason
0,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,2.386392e-11,0.041549,14.559809,814.967844,...,862960.422402,1058.942911,False,False,False,False,False,False,False,
1,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,420,8.940858e-01,0.982250,1.030267,1.174808,...,9.301545,48.305272,False,False,False,False,False,False,False,
2,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,391,9.204675e-01,0.979163,1.025078,1.157933,...,8.821935,49.347799,False,False,False,False,False,False,False,
3,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.665369e+00,2.931296,6.948123,23.066052,...,224.175002,11.133733,False,False,False,False,False,False,False,
4,pyspoc.statistics.basic.Covariance.feature_cov...,1044,0,0.0,1044,1044,1.618522e+00,2.354643,5.024400,14.776044,...,162.795351,13.106037,False,False,False,False,False,False,False,


In [30]:
# ============================================================
# Stage 2.8 Inspect remaining missing values
# ============================================================

missing_by_feature = (
    features_filtered_raw
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_by_feature = missing_by_feature[missing_by_feature > 0]

print("Number of kept features with at least one missing value:", len(missing_by_feature))
print("Total remaining missing values:", int(missing_by_feature.sum()))

display(missing_by_feature.head(30).to_frame("missing_count"))

Number of kept features with at least one missing value: 0
Total remaining missing values: 0


,missing_count


In [31]:
missing_by_dataset = (
    features_filtered_raw
    .isna()
    .sum(axis=1)
    .sort_values(ascending=False)
)

missing_by_dataset = missing_by_dataset[missing_by_dataset > 0]

print("Number of datasets with at least one missing feature:", len(missing_by_dataset))
print("Total remaining missing values:", int(missing_by_dataset.sum()))

display(
    pd.concat(
        [
            metadata_df.loc[missing_by_dataset.index, [
                "dataset_id",
                "file_stem",
                "model_family",
                "n_clusters",
                "contrast_level",
                "shape_name",
                "seed",
            ]],
            missing_by_dataset.rename("missing_feature_count")
        ],
        axis=1
    ).head(30)
)

Number of datasets with at least one missing feature: 0
Total remaining missing values: 0


,dataset_id,file_stem,model_family,n_clusters,contrast_level,shape_name,seed,missing_feature_count


In [32]:
# ============================================================
# Stage 2.9 Drop the remaining missing feature
# ============================================================

features_with_any_missing = features_filtered_raw.columns[
    features_filtered_raw.isna().any()
].tolist()

print("Features with any remaining missing values:")
for feature in features_with_any_missing:
    print(feature)

features_filtered_raw_no_missing = features_filtered_raw.drop(
    columns=features_with_any_missing
)

print("Shape before dropping remaining missing features:", features_filtered_raw.shape)
print("Shape after dropping remaining missing features:", features_filtered_raw_no_missing.shape)
print("Remaining missing values:", features_filtered_raw_no_missing.isna().sum().sum())

Features with any remaining missing values:
Shape before dropping remaining missing features: (1044, 1214)
Shape after dropping remaining missing features: (1044, 1214)
Remaining missing values: 0
